# TechMind - Organización Inteligente del Conocimiento Técnico
## Auditoría, limpieza y enriquecimiento inicial del dataset

Este notebook realiza la auditoría, limpieza y enriquecimiento inicial del dataset del proyecto TechMind.
El objetivo es entender el estado real de los datos, identificar ruido, duplicados, valores nulos y características del texto,
para luego generar una versión limpia y estructurada que sirva como base para módulos de clasificación, búsqueda semántica y reutilización del conocimiento técnico.

##Importaciones y carga

In [3]:
import pandas as pd
import numpy as np
import re
import html
import uuid
!pip install langdetect
from langdetect import detect, LangDetectException

In [4]:
df = pd.read_csv("dataset_techmind_original.csv")
df.columns = [c.strip().lower() for c in df.columns]

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
display(df.head())

Filas: 1008
Columnas: 3


,source_type,is_synthetic_noise,raw_text
0,Tutorial_Codigo,False,Contexto: Search for a text pattern in multipl...
1,Tutorial_Codigo,False,Contexto: Profile code execution time: {code}\...
2,Tutorial_Codigo,False,Contexto: Send a weekly scheduled email with t...
3,Articulo_Academico,False,"In this paper, we consider the nonasymptotic..."
4,Tutorial_Codigo,False,Contexto: List all running Docker containers\n...


##Auditoria

In [5]:
print("=== ESTRUCTURA GENERAL ===")
print("Dimensión:", df.shape)
print("Columnas:", list(df.columns))
print()

print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print()

print("=== VALORES NULOS ===")
print(df.isnull().sum().sort_values(ascending=False))
print()

print("=== DUPLICADOS ===")
print("Filas duplicadas:", df.duplicated().sum())
print()

print("=== MUESTRA DE REGISTROS ===")
display(df.sample(5, random_state=42))

=== ESTRUCTURA GENERAL ===
Dimensión: (1008, 3)
Columnas: ['source_type', 'is_synthetic_noise', 'raw_text']

=== TIPOS DE DATOS ===
source_type           object
is_synthetic_noise      bool
raw_text              object
dtype: object

=== VALORES NULOS ===
source_type           0
is_synthetic_noise    0
raw_text              0
dtype: int64

=== DUPLICADOS ===
Filas duplicadas: 0

=== MUESTRA DE REGISTROS ===


,source_type,is_synthetic_noise,raw_text
938,Articulo_Academico,False,We study the problem of partitioning a small...
630,Articulo_Academico,False,"In this paper, we propose a special fusion m..."
682,Articulo_Academico,False,We show that Boolean functions expressible a...
514,Tutorial_Codigo,False,Contexto: Open Google Maps if I say 'I'm lost'...
365,Tutorial_Codigo,False,Contexto: Monitor the health of all hard drive...


In [6]:
# Detectar columna principal de texto
text_candidates = [c for c in df.columns if df[c].dtype == "object"]
print("Columnas de tipo texto:", text_candidates)

text_col = "rawtext" if "rawtext" in df.columns else (text_candidates[-1] if text_candidates else None)
print("Columna de texto usada:", text_col)

Columnas de tipo texto: ['source_type', 'raw_text']
Columna de texto usada: raw_text


In [7]:
if text_col:
    df[text_col] = df[text_col].fillna("").astype(str)
    df["text_length_chars"] = df[text_col].apply(len)
    df["text_length_words"] = df[text_col].apply(lambda x: len(x.split()))

    print("=== LONGITUD DEL TEXTO ===")
    print("Promedio caracteres:", round(df["text_length_chars"].mean(), 2))
    print("Mediana caracteres:", round(df["text_length_chars"].median(), 2))
    print("Promedio palabras:", round(df["text_length_words"].mean(), 2))
    print("Mediana palabras:", round(df["text_length_words"].median(), 2))

=== LONGITUD DEL TEXTO ===
Promedio caracteres: 683.87
Mediana caracteres: 477.0
Promedio palabras: 93.98
Mediana palabras: 59.0


In [8]:
# Distribución de posibles categorías
possible_category_cols = [c for c in ["sourcetype", "category", "label", "type", "class"] if c in df.columns]

print("=== COLUMNAS CATEGÓRICAS POSIBLES ===")
print(possible_category_cols)

for col in possible_category_cols:
    print(f"\nDistribución de {col}:")
    print(df[col].value_counts(dropna=False))

=== COLUMNAS CATEGÓRICAS POSIBLES ===
[]


In [9]:
# Ruido básico
if text_col:
    urls_count = df[text_col].str.contains(r"http[s]?://|www\.", regex=True, na=False).sum()
    html_count = df[text_col].str.contains(r"<[^>]+>", regex=True, na=False).sum()

    special_chars = df[text_col].apply(lambda x: len(re.findall(r"[^\w\sáéíóúÁÉÍÓÚñÑüÜ.,;:()\-\/]", x)))

    print("=== RUIDO BÁSICO ===")
    print("Filas con URLs:", urls_count)
    print("Filas con HTML:", html_count)
    print("Promedio de caracteres especiales raros:", round(special_chars.mean(), 2))

=== RUIDO BÁSICO ===
Filas con URLs: 107
Filas con HTML: 11
Promedio de caracteres especiales raros: 18.27


##Limpieza

In [10]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [11]:
df_clean = df.copy()

# Eliminar filas vacías y duplicados
df_clean = df_clean.dropna(how="all")
df_clean = df_clean.drop_duplicates()

# Limpiar texto
if text_col:
    df_clean[f"{text_col}_clean"] = df_clean[text_col].apply(clean_text)
    df_clean[f"{text_col}_clean"] = df_clean[f"{text_col}_clean"].str.replace(r"\s+", " ", regex=True).str.strip()

    # Eliminar textos vacíos
    df_clean = df_clean[df_clean[f"{text_col}_clean"].str.len() > 0].copy()

print("Filas después de limpieza:", df_clean.shape[0])
display(df_clean.head())

Filas después de limpieza: 1008


,source_type,is_synthetic_noise,raw_text,text_length_chars,text_length_words,raw_text_clean
0,Tutorial_Codigo,False,Contexto: Search for a text pattern in multipl...,268,42,Contexto: Search for a text pattern in multipl...
1,Tutorial_Codigo,False,Contexto: Profile code execution time: {code}\...,234,27,Contexto: Profile code execution time: {code} ...
2,Tutorial_Codigo,False,Contexto: Send a weekly scheduled email with t...,524,49,Contexto: Send a weekly scheduled email with t...
3,Articulo_Academico,False,"In this paper, we consider the nonasymptotic...",925,147,"In this paper, we consider the nonasymptotic s..."
4,Tutorial_Codigo,False,Contexto: List all running Docker containers\n...,125,17,Contexto: List all running Docker containers C...


##Enriquecimiento

In [12]:
def detect_language_safe(text):
    try:
        if not text or len(text.strip()) < 10:
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"
    except Exception:
        return "unknown"

In [13]:
if text_col:
    df_clean["doc_id"] = [str(uuid.uuid4()) for _ in range(len(df_clean))]
    df_clean["language"] = df_clean[f"{text_col}_clean"].apply(detect_language_safe)
    df_clean["clean_length_chars"] = df_clean[f"{text_col}_clean"].str.len()
    df_clean["clean_length_words"] = df_clean[f"{text_col}_clean"].str.split().apply(len)

display(df_clean[[col for col in ["doc_id", "language", "clean_length_chars", "clean_length_words"]
        if col in df_clean.columns]].head())

,doc_id,language,clean_length_chars,clean_length_words
0,023555c4-83e3-493a-9ea5-9461289b02db,en,267,42
1,d2d6d4e9-c417-4555-8a3b-74d3c2bf2d23,en,233,27
2,d92f4e8e-8f01-4347-b981-26312a6fc5c2,en,491,49
3,bde8c5ea-8384-42f4-bb7d-65ff4cdb3de3,en,922,147
4,9a0ee165-60a4-48ba-b8c9-d7043643ddc7,en,124,17


In [14]:
print("=== DISTRIBUCIÓN DE IDIOMA ===")
print(df_clean["language"].value_counts(dropna=False))

=== DISTRIBUCIÓN DE IDIOMA ===
language
en    975
fr     23
es      8
ca      2
Name: count, dtype: int64


##Persistencia

In [15]:
df_clean.to_csv("dataset_techmind_clean.csv", index=False)
print("Archivo guardado: dataset_techmind_clean.csv")

Archivo guardado: dataset_techmind_clean.csv


##Control Final

In [16]:
print("=== RESUMEN FINAL ===")
print("Registros originales:", df.shape[0])
print("Registros limpios:", df_clean.shape[0])
print("Columnas finales:", df_clean.shape[1])
print("Duplicados eliminados:", df.duplicated().sum())
print("Nulos originales por columna:")
print(df.isnull().sum().sort_values(ascending=False))

=== RESUMEN FINAL ===
Registros originales: 1008
Registros limpios: 1008
Columnas finales: 10
Duplicados eliminados: 0
Nulos originales por columna:
source_type           0
is_synthetic_noise    0
raw_text              0
text_length_chars     0
text_length_words     0
dtype: int64
